# Part 1 — Rebuilding the GEAR vs reality table

Nattrass (2001) compared GEAR's projections with what actually happened between
1996 and 1999. This notebook rebuilds that table for the six indicators the
assignment asks for, corrects one error in the supplied data, and fills in 2000.

**Output:** two CSV files in `output/data/`.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd

# Find the project folder: walk up from the working directory until we see a
# folder holding both resources/ and output/. This works whether the notebook is
# run from its own folder or from the repository root, so it keeps working after
# the project is moved or cloned onto another machine.
def find_project(start=None):
    here = Path(start or Path.cwd()).resolve()
    for folder in [here, *here.parents]:
        if (folder / "resources").is_dir() and (folder / "output").is_dir():
            return folder
    raise FileNotFoundError(
        "Could not find the project folder (the one containing 'resources' and "
        "'output'). Open this notebook from inside the project folder."
    )


PROJECT = find_project()

RES = PROJECT / "resources"                  # supplied data, never written to
BASE = PROJECT / "output"                    # everything we generate

print("Project folder:", PROJECT)

## Load the supplied data

The file is in long format: one row per indicator per year.

In [ ]:
# Read the GEAR targets and outturns supplied with the assignment.
RAW_CSV = RES / "data" / "gear_target_vs_actual_1996_2000 (1).csv"
gear = pd.read_csv(RAW_CSV)

print(f"{len(gear)} rows, {gear['indicator'].nunique()} indicators")
gear.head()

## Correction 1 — the inflation targets are from the wrong table

The supplied file gives GEAR inflation targets of 8.4, 9.7 and 9.6 for 1996-98.
Those come from GEAR's **Base Scenario**, the do-nothing baseline the strategy was
designed to beat, not the **Integrated Scenario** that government actually adopted
and that every other row uses.

Checked against South Africa (1996), the Integrated Scenario gives 8.0, 9.7, 8.1.
Only 1999 (7.7) happens to match in both scenarios, which is probably why the
mistake went unnoticed. Nattrass's own 2001 table repeats the same wrong figures,
so this is an error inherited from the published paper.

In [ ]:
# Correct inflation targets, from South Africa (1996) Integrated Scenario table.
CORRECTED_INFLATION = {1996: 8.0, 1997: 9.7, 1998: 8.1, 1999: 7.7, 2000: 7.6}

before = gear.loc[(gear["indicator"] == "inflation_cpi_pct")
                  & (gear["year"].between(1996, 1998)), "gear_target"].tolist()

for year, value in CORRECTED_INFLATION.items():
    mask = (gear["indicator"] == "inflation_cpi_pct") & (gear["year"] == year)
    if mask.any():
        gear.loc[mask, "gear_target"] = value
    else:
        # 2000 was blank in the supplied file, so add the row.
        gear = pd.concat([gear, pd.DataFrame([{
            "indicator": "inflation_cpi_pct", "year": year,
            "gear_target": value, "actual_to_1999": pd.NA,
            "scan_reconstructed": pd.NA,
        }])], ignore_index=True)

print("supplied 1996-98:", before)
print("corrected 1996-2000:", [CORRECTED_INFLATION[y] for y in range(1996, 2001)])

## Correction 2 — fill in the year 2000

Nattrass stops at 1999 because her source (SARB *Quarterly Bulletin*, March 2000)
predates the full-year figures. These come from the SARB Annual Economic Report
2001, the first report covering all of calendar 2000.

Employment is deliberately left blank. The report gives private-sector formal
employment falling 2.0% and public-sector falling 4.1%, but never a combined
total, and building one would need a weighting we cannot source. A blank is
honest; an invented number is not.

In [ ]:
# 2000 actuals from SARB Annual Economic Report 2001.
ACTUAL_2000 = {
    "real_gdp_growth_pct": 3.0,
    "real_private_investment_growth_pct": 5.0,
    "inflation_cpi_pct": 5.3,
    "deficit_pct_gdp": 2.0,     # fiscal year 2000/01, not calendar year
    "real_bank_rate_pct": 6.51,  # 11.81 nominal minus 5.3 CPI
}

for indicator, value in ACTUAL_2000.items():
    mask = (gear["indicator"] == indicator) & (gear["year"] == 2000)
    gear.loc[mask, "actual_to_1999"] = value

for k, v in ACTUAL_2000.items():
    print(f"{k:40s} {v}")

## Check the cells reconstructed from a damaged scan

Cells flagged `R` were reconstructed by the course compiler from a poor scan.
Each was checked against Nattrass (2001) and the 1996 GEAR document. All match
the supplied values, so none needed changing.

Note the data dictionary says six cells are affected, but only five carry the flag.

In [ ]:
# Show every cell flagged as reconstructed from the scan.
flagged = gear[gear["scan_reconstructed"] == "R"]
print(f"{len(flagged)} flagged cells (the data dictionary claims six)")
flagged[["indicator", "year", "gear_target", "actual_to_1999"]]

## Build the table

Reshape to one row per indicator, with a target and actual column for each year.

In [ ]:
# The six indicators the assignment asks for, in order.
REQUIRED = [
    "real_gdp_growth_pct",
    "real_private_investment_growth_pct",
    "nonagric_formal_employment_growth_pct",
    "inflation_cpi_pct",
    "deficit_pct_gdp",
    "real_bank_rate_pct",
]

subset = gear[gear["indicator"].isin(REQUIRED)]

# Pivot to wide: indicators down the side, years across the top.
table = subset.pivot_table(index="indicator", columns="year",
                           values=["gear_target", "actual_to_1999"],
                           aggfunc="first")
table = table.swaplevel(axis=1).sort_index(axis=1, level=0).reindex(REQUIRED)
table

## Save

In [ ]:
# Save the corrected long file and the wide table.
(BASE / "data").mkdir(parents=True, exist_ok=True)
gear.to_csv(BASE / "data" / "gear_target_vs_actual_1996_2000_CORRECTED.csv", index=False)
table.to_csv(BASE / "data" / "part1_gear_vs_reality_table.csv")

print("Saved to", BASE / "data")

## What this shows

Growth, investment and employment miss their targets in every year. Inflation and
the deficit beat theirs. GEAR delivered the stability it promised and none of the
growth that stability was supposed to buy.